# MNIST Digit Classification — Fully Connected Network (Keras)

A short, clean Keras/TensorFlow pipeline that classifies handwritten digits
(0-9) from MNIST with a plain fully connected network.

The **Config** below starts deliberately weak (a tiny net, trained briefly), so
it scores only ~90%. Tune the knobs, re-run the Model and Train cells, and
push the test accuracy up toward ~98%.

## Imports

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)
print("TensorFlow", tf.__version__)

## Config (hyperparameters)

In [ ]:
@dataclass
class Config:
    """All tunable knobs in one place. Defaults start weak on purpose."""

    hidden_layers: int = 1            # depth: start with 1 - try 2, 3
    hidden_units: int = 16            # width: start tiny - widen to 256, 512
    activation: str = "relu"          # try https://www.tensorflow.org/api_docs/python/tf/keras/activations
    loss: str = "mse"  # try "categorical_crossentropy", "mse"
    optimizer: str = "adam"           # try https://www.tensorflow.org/api_docs/python/tf/keras/optimizers
    epochs: int = 2                   # start short - raise to 10-15
    batch_size: int = 256             # try 32, 64, 128
    learning_rate: float = 1e-3       # try 1e-2 ... 1e-4


cfg = Config()
cfg

## Data

Load MNIST and scale pixels to `[0, 1]`. Labels stay as integers 0-9; the
**Train** cell one-hot encodes them only when the chosen loss needs it.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

print("train:", x_train.shape, "| test:", x_test.shape)

## Model

Flatten the 28x28 image, then `cfg.hidden_layers` hidden Dense layers of
`cfg.hidden_units` neurons each, then a softmax output with one unit per digit
class.

In [ ]:
NUM_CLASSES = 10  # digits 0-9 (fixed by the task, not a hyperparameter)


def build_model(cfg: Config) -> tf.keras.Model:
    """Flatten -> cfg.hidden_layers hidden Dense layers -> Dense(NUM_CLASSES, softmax)."""
    hidden = [
        layers.Dense(cfg.hidden_units, activation=cfg.activation)
        for _ in range(cfg.hidden_layers)
    ]
    return models.Sequential(
        [
            layers.Input(shape=(28, 28)),
            layers.Flatten(),
            *hidden,
            layers.Dense(NUM_CLASSES, activation="softmax"),
        ]
    )


model = build_model(cfg)
model.summary()

## Train

Build the optimizer from `cfg.optimizer` (honoring `cfg.learning_rate`), then
train with the loss from `cfg.loss`. `"mse"` and `"categorical_crossentropy"`
compare against one-hot targets, so we encode the labels to match; sparse
losses take the integers directly.

In [ ]:
OPTIMIZERS = {
    "adam": tf.keras.optimizers.Adam,
    "sgd": tf.keras.optimizers.SGD,
    "rmsprop": tf.keras.optimizers.RMSprop,
    "adamw": tf.keras.optimizers.AdamW,
    "nadam": tf.keras.optimizers.Nadam,
}
LOSSES_NEEDING_ONE_HOT = {"categorical_crossentropy", "mse", "mean_squared_error"}


def build_optimizer(cfg: Config) -> tf.keras.optimizers.Optimizer:
    """Look up the optimizer by name and set its learning rate."""
    if cfg.optimizer not in OPTIMIZERS:
        raise ValueError(f"Unknown optimizer {cfg.optimizer!r}. Choose from {list(OPTIMIZERS)}.")
    return OPTIMIZERS[cfg.optimizer](learning_rate=cfg.learning_rate)


def encode_labels(labels):
    """One-hot the labels if the chosen loss needs it, else leave them as integers."""
    if cfg.loss in LOSSES_NEEDING_ONE_HOT:
        return tf.one_hot(labels, NUM_CLASSES)
    return labels


model.compile(optimizer=build_optimizer(cfg), loss=cfg.loss, metrics=["accuracy"])

history = model.fit(
    x_train,
    encode_labels(y_train),
    epochs=cfg.epochs,
    batch_size=cfg.batch_size,
    validation_data=(x_test, encode_labels(y_test)),
)

## Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, encode_labels(y_test), verbose=0)
print(f"Test accuracy: {test_acc * 100:.2f}%  |  loss: {test_loss:.4f}")

## Results & visualization

In [ ]:
def plot_history(history: tf.keras.callbacks.History) -> None:
    """Plot loss and accuracy curves side by side."""
    h = history.history
    epochs = range(1, len(h["loss"]) + 1)
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))

    ax_loss.plot(epochs, h["loss"], label="train")
    ax_loss.plot(epochs, h["val_loss"], label="test")
    ax_loss.set_title("Loss")
    ax_loss.set_xlabel("epoch")
    ax_loss.legend()

    ax_acc.plot(epochs, h["accuracy"], label="train")
    ax_acc.plot(epochs, h["val_accuracy"], label="test")
    ax_acc.set_title("Accuracy")
    ax_acc.set_xlabel("epoch")
    ax_acc.legend()

    plt.tight_layout()
    plt.show()


plot_history(history)

In [ ]:
def show_predictions(model, images, labels, n: int = 8) -> None:
    """Display n test images with predicted vs. true labels."""
    preds = model.predict(images[:n], verbose=0).argmax(axis=1)

    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.5))
    for ax, image, pred, true in zip(axes, images[:n], preds, labels[:n]):
        ax.imshow(image, cmap="gray")
        color = "green" if pred == true else "red"
        ax.set_title(f"pred {pred} / true {true}", color=color, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_predictions(model, x_test, y_test)